# DeepExoMir quickstart

Three things this notebook covers:

1. Loading the DeepExoMir-Lite checkpoint (recommended production variant).
2. Scoring a single (miRNA, target window) pair.
3. Reproducing the Table 1 benchmark on the three miRBench test sets.

Approximate runtime: ~10 minutes on a single CPU core, ~90 seconds on an RTX 5090.

All API calls in this notebook are real and exist in the `deepexomir/` package; no pseudo-code.

## 1. Dependencies

Install via `pip install -e .` from the repository root.  The notebook also depends on `miRBench` (auto-installed via `pip install miRBench` or via the project Dockerfile).

In [ ]:
import torch
import numpy as np
from pathlib import Path

from deepexomir.config import load_config
from deepexomir.predict import load_model, score_pair, score_batch, load_pca
from deepexomir.benchmark import evaluate_mirbench_test_sets, summarize_results

print(f'PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')

## 2. Load DeepExoMir-Lite

Lite has approximately 35% fewer parameters than v19, no ViennaRNA dependency, and equivalent or marginally better mean test AU-PRC (0.863 vs 0.855).  We use the best validation checkpoint; the path below is the actual file as released on Zenodo.

In [ ]:
REPO_ROOT = Path('..').resolve()
config = load_config(REPO_ROOT / 'configs/model_config_v19_noStructure.yaml')
checkpoint = REPO_ROOT / 'checkpoints/v19_noStructure/checkpoint_epoch010_val_auc_0.8238.pt'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, backbone = load_model(checkpoint=checkpoint, config=config, load_backbone=True, device=device)

n_params = sum(p.numel() for p in model.parameters())
print(f'DeepExoMir-Lite loaded: {n_params/1e6:.1f}M parameters on {device}')

## 3. Score one (miRNA, target window) pair

Inputs are two RNA strings (Watson-Crick alphabet AUCG).  For best results pass a 30-50 nt 3'UTR window centered on the candidate site, but the model handles any length up to its training maximum.  The returned probability is best interpreted as a *ranking* signal -- see Section 2.6 of the paper for the calibration analysis and the recommended percentile thresholds.

In [ ]:
mirna_seq  = 'GUGAAAUGUUUAGGACCACUAG'                            # hsa-miR-203a-3p
target_seq = 'AGGCUUAUGCAUUUCAGAUUUCAGAGAUUUUCAGAUUUUCAGAGAUU'  # KITLG 3'UTR window

score = score_pair(model, backbone, mirna_seq, target_seq)
print(f'Predicted score: {score:.4f}')

if score > 0.7:
    interp = 'high-confidence target site'
elif score > 0.4:
    interp = 'plausible candidate'
else:
    interp = 'unlikely'
print(f'Interpretation: {interp}')

## 4. Reproduce the Table 1 benchmark

Evaluates DeepExoMir-Lite on three miRBench held-out test sets:
* AGO2 CLASH Hejret 2023 (n = 965)
* AGO2 eCLIP Klimentová 2022 (n = 954)
* AGO2 eCLIP Manakov 2022 (n = 327,129)

Expected mean AU-PRC for Lite is 0.863 (matches the paper Table 1 footnote).

In [ ]:
results = evaluate_mirbench_test_sets(model, backbone, batch_size=128)
for ds, m in results.items():
    print(f'{ds:<28}  AU-PRC={m["au_prc"]:.4f}  ROC-AUC={m["roc_auc"]:.4f}  n={m["n"]}')

summary = summarize_results(results)
print(f'\nMean AU-PRC across {summary["n_datasets"]} miRBench test sets: {summary["mean_au_prc"]:.4f}')

## Next steps

* Reproduce **Table 3** (retrain-from-scratch ablation): `python scripts/reproduce_table3.py --output results/table3.tsv`.  This reads pre-computed score files under `manuscript/BiB_submission/baseline_scores/` and runs paired sample-level bootstrap.
* Regenerate the **calibration figure** (Figure 10): `python manuscript/BiB_submission/scripts/calibration_analysis.py`.
* Score a transcriptome-scale set (one miRNA × ~19,000 3'UTRs): see `scripts/predict.py` for the leakage-tier-aware retrospective scoring pipeline.

Code: https://github.com/linwenh09/DeepExoMir  
Archive: https://doi.org/10.5281/zenodo.19216306